# Zero-Cost NAS Lab: NASWOT and SynFlow

In [4]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/nhdnhut/naswot-eznas-synflow-nb.git"
REPO_DIR = Path("/content/naswot-eznas-synflow-nb")

# Khi notebook được mở trong chính repo (Jupyter/local), không cần clone lại.
if not (Path.cwd() / "src" / "zero_cost_metrics.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())
print("Metrics module exists:", (Path.cwd() / "src" / "zero_cost_metrics.py").exists())


Working directory: /content/naswot-eznas-synflow-nb
Metrics module exists: True


In [5]:
import copy
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from scipy.stats import kendalltau, spearmanr

from src.zero_cost_metrics import (
    count_parameters,
    naswot_score,
    synflow_score,
    synflow_iterative_prune,
)

print("Python / PyTorch environment")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python / PyTorch environment
PyTorch: 2.14.1+cpu
Torchvision: 0.29.1+cpu
NumPy: 2.4.6
Pandas: 2.2.3
CUDA available: False


track seed

In [6]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 2


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
print("Device:", DEVICE)
print("Seed:", SEED)


Device: cpu
Seed: 42


mini-batch CIFAR-10

In [7]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

images, labels = next(iter(train_loader))
images = images.to(DEVICE)
labels = labels.to(DEVICE)

print("Input batch:", tuple(images.shape))
print("Labels are loaded but are not used by NASWOT:", tuple(labels.shape))


100%|██████████| 170M/170M [21:45<00:00, 131kB/s]  


Input batch: (64, 3, 32, 32)
Labels are loaded but are not used by NASWOT: (64,)


pipeline

In [8]:
class SmallCNN(nn.Module):
    def __init__(self, channels: tuple[int, ...], num_classes: int = 10):
        super().__init__()
        if not channels:
            raise ValueError("channels must not be empty")

        layers = []
        in_channels = 3
        for out_channels in channels:
            layers.extend([
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=True),
                nn.ReLU(inplace=False),
                nn.MaxPool2d(kernel_size=2),
            ])
            in_channels = out_channels

        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(in_channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


ARCHITECTURES = {
    "cnn_small": (16, 32),
    "cnn_wide": (24, 48),
    "cnn_deeper": (16, 24, 32),
}

for name, channels in ARCHITECTURES.items():
    model = SmallCNN(channels).to(DEVICE)
    print(f"{name:12s} channels={channels!s:14s} params={count_parameters(model):,}")


cnn_small    channels=(16, 32)       params=5,418
cnn_wide     channels=(24, 48)       params=11,578
cnn_deeper   channels=(16, 24, 32)   params=11,202


naswot and synflow score

In [9]:
rows = []
models = {}

for index, (name, channels) in enumerate(ARCHITECTURES.items()):
    architecture_seed = 1000 + index
    set_seed(architecture_seed)
    model = SmallCNN(channels).to(DEVICE)
    models[name] = model

    nas_result = naswot_score(model, images, jitter=1e-6)
    syn_result = synflow_score(
        model,
        input_shape=(1, 3, 32, 32),
        device=DEVICE,
        use_double=False,
    )

    rows.append({
        "architecture": name,
        "channels": str(channels),
        "seed": architecture_seed,
        "parameters": count_parameters(model),
        "naswot_score": nas_result["score"],
        "naswot_raw_logdet": nas_result["raw_logdet"],
        "naswot_sign": nas_result["sign"],
        "naswot_num_relu_units": nas_result["num_relu_units"],
        "synflow_score": syn_result["score"],
        "synflow_scored_parameters": syn_result["scored_parameter_count"],
    })

scores_df = pd.DataFrame(rows)
scores_df


,architecture,channels,seed,parameters,naswot_score,naswot_raw_logdet,naswot_sign,naswot_num_relu_units,synflow_score,synflow_scored_parameters
0,cnn_small,"(16, 32)",1000,5418,576.577806,576.577806,1,24576,1435.370646,5418
1,cnn_wide,"(24, 48)",1001,11578,600.422730,600.422730,1,36864,1970.864799,11578
2,cnn_deeper,"(16, 24, 32)",1002,11202,570.101265,570.101265,1,24576,12978.855510,11202


prunning

In [10]:
pruning_result = synflow_iterative_prune(
    models["cnn_small"],
    input_shape=(1, 3, 32, 32),
    compression_ratio=2.0,
    steps=5,  # smoke test; paper reports 100 iterations for its pruning experiments
    device=DEVICE,
    use_double=False,
)

pruning_history_df = pd.DataFrame(pruning_result["history"])
print("Achieved sparsity:", pruning_result["sparsity"])
print("Remaining / total:", pruning_result["remaining_parameters"], "/", pruning_result["total_parameters"])
pruning_history_df


Achieved sparsity: 0.5
Remaining / total: 2709 / 5418


,step,percentile,threshold,remaining_parameters,sparsity
0,1,12.944944,0.029951,4716,0.129568
1,2,24.214172,0.054588,4106,0.242156
2,3,34.024604,0.076717,3574,0.340347
3,4,42.565082,0.094975,3112,0.425618
4,5,50.000000,0.110292,2709,0.500000


batch size diverse

In [11]:
reference_model = models["cnn_small"]
batch_rows = []

for batch_size in [8, 16, 32, 64]:
    result = naswot_score(
        reference_model,
        images[:batch_size],
        jitter=1e-6,
    )
    batch_rows.append({
        "batch_size": batch_size,
        "score_with_jitter": result["score"],
        "raw_logdet": result["raw_logdet"],
        "sign": result["sign"],
        "num_relu_units": result["num_relu_units"],
    })

batch_sensitivity_df = pd.DataFrame(batch_rows)
batch_sensitivity_df


,batch_size,score_with_jitter,raw_logdet,sign,num_relu_units
0,8,76.423204,76.423204,1,24576
1,16,149.215394,149.215394,1,24576
2,32,292.776394,292.776394,1,24576
3,64,576.577806,576.577806,1,24576


In [12]:
seed_rows = []
SEEDS = [11, 22, 33, 44, 55]

for name, channels in ARCHITECTURES.items():
    for seed in SEEDS:
        set_seed(seed)
        model = SmallCNN(channels).to(DEVICE)
        nas_result = naswot_score(model, images, jitter=1e-6)
        syn_result = synflow_score(model, (1, 3, 32, 32), device=DEVICE)
        seed_rows.append({
            "architecture": name,
            "seed": seed,
            "parameters": count_parameters(model),
            "naswot_score": nas_result["score"],
            "synflow_score": syn_result["score"],
        })

seed_results_df = pd.DataFrame(seed_rows)
seed_summary_df = seed_results_df.groupby("architecture").agg(
    naswot_mean=("naswot_score", "mean"),
    naswot_std=("naswot_score", "std"),
    synflow_mean=("synflow_score", "mean"),
    synflow_std=("synflow_score", "std"),
    parameters=("parameters", "first"),
).reset_index()
seed_summary_df


,architecture,naswot_mean,naswot_std,synflow_mean,synflow_std,parameters
0,cnn_deeper,571.923562,3.611765,13289.374442,660.758819,11202
1,cnn_small,572.492876,3.159971,1319.361783,28.848720,5418
2,cnn_wide,599.110405,3.249634,1985.533522,58.973565,11578


In [13]:
def rank_correlations(
    frame: pd.DataFrame,
    target_column: str = "val_acc",
    proxy_columns: tuple[str, ...] = ("naswot_score", "synflow_score"),
) -> pd.DataFrame:
    """Compute Spearman rho and Kendall tau-b against a ground-truth target."""
    required = [target_column, *proxy_columns]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(
            f"Missing columns {missing}. Add benchmark/training accuracy first; "
            "do not fabricate target values."
        )

    output = []
    for proxy in proxy_columns:
        valid = frame[[proxy, target_column]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(valid) < 2:
            raise ValueError(f"At least two valid rows are required for {proxy}")
        rho = spearmanr(valid[proxy], valid[target_column])
        tau = kendalltau(valid[proxy], valid[target_column], variant="b")
        output.append({
            "proxy": proxy,
            "n_architectures": len(valid),
            "spearman_rho": float(rho.statistic),
            "kendall_tau_b": float(tau.statistic),
        })
    return pd.DataFrame(output)

# Khi đã có bảng benchmark, dùng:
# correlation_df = rank_correlations(benchmark_results_df, target_column="val_acc")
# display(correlation_df)
print("Correlation helper is ready; provide real validation accuracy to use it.")


Correlation helper is ready; provide real validation accuracy to use it.


In [ ]:
def train_and_evaluate(model_name, channels, epochs=2, lr=0.01):
    print(f"Training {model_name} for {epochs} epochs to obtain real val_acc...")
    set_seed(42)
    model = SmallCNN(channels).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    # Prepare a validation loader (using 20% of CIFAR-10 training set for speed)
    val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

    # Training loop
    model.train()
    for epoch in range(epochs):
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # Limit batches per epoch to speed up training for teaching experiment
            if batch_idx > 150:
                break
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

    # Validation accuracy evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_acc = 100.0 * correct / total
    print(f"Finished {model_name}. Validation Accuracy: {val_acc:.2f}%")
    return val_acc

# Run training and fetch actual accuracies
ground_truth_accuracies = {}
for name, channels in ARCHITECTURES.items():>
    val_acc = train_and_evaluate(name, channels, epochs=2)
    ground_truth_accuracies[name] = val_acc

Training cnn_small for 2 epochs to obtain real val_acc...
Finished cnn_small. Validation Accuracy: 25.70%
Training cnn_wide for 2 epochs to obtain real val_acc...
Finished cnn_wide. Validation Accuracy: 25.55%
Training cnn_deeper for 2 epochs to obtain real val_acc...
Finished cnn_deeper. Validation Accuracy: 27.53%


In [15]:
# Map ground-truth validation accuracy to the summary DataFrame from Lab B
benchmark_results_df = seed_summary_df.copy()
benchmark_results_df["val_acc"] = benchmark_results_df["architecture"].map(ground_truth_accuracies)

# Rename columns to match rank_correlations input expectation if necessary
benchmark_results_df = benchmark_results_df.rename(columns={
    "naswot_mean": "naswot_score",
    "synflow_mean": "synflow_score"
})

display(benchmark_results_df)

,architecture,naswot_score,naswot_std,synflow_score,synflow_std,parameters,val_acc
0,cnn_deeper,571.923562,3.611765,13289.374442,660.758819,11202,27.53
1,cnn_small,572.492876,3.159971,1319.361783,28.848720,5418,25.70
2,cnn_wide,599.110405,3.249634,1985.533522,58.973565,11578,25.55


In [16]:
# Compute Spearman rho and Kendall tau-b rank correlations
correlation_df = rank_correlations(benchmark_results_df, target_column="val_acc")
display(correlation_df)

,proxy,n_architectures,spearman_rho,kendall_tau_b
0,naswot_score,3,-1.0,-1.000000
1,synflow_score,3,0.5,0.333333
